# META-CXR Training on GCP Compute Engine (2x T4 spot)


In [ ]:
# === Papermill parameters cell ===
# Override these from CLI: papermill ... -p RUN_ID my-run -p RESUME_FROM checkpoint_last.pth
RUN_ID = "mimic_cxr_2gpu_stage1"
RESUME_FROM = ""        # "" = auto-pick best.pth from GCS; or "checkpoint_last.pth" / "checkpoint_N.pth"
BATCH_SIZE = None       # None = giữ value trong configs/training.yaml
MAX_EPOCH = None        # None = giữ value trong configs/training.yaml
SMOKE_TEST = False      # True = override max_epoch=1, batch_size=2 cho smoke run

import os, sys
os.environ.setdefault("RUN_ID", RUN_ID)

# Install SIGTERM flush handler (spot eviction safety).
sys.path.insert(0, "/workspace/Meta-CXR-GCP")
try:
    import yaml
    from src.gcs_checkpoint import install_sigterm_flush
    with open("/workspace/Meta-CXR-GCP/configs/checkpoint.yaml") as _f:
        _CKPT = yaml.safe_load(_f)
    install_sigterm_flush(
        run_id=RUN_ID,
        local_dir=_CKPT["local_dir"],
        gcs_prefix=_CKPT["gcs_prefix"],
        preserve_files=_CKPT["preserve_files"],
        keep_last_n=_CKPT["keep_last_n_epoch_ckpts"],
        flush_seconds=_CKPT["upload"]["sigterm_flush_seconds"],
    )
    print("SIGTERM flush handler installed.")
except Exception as _e:
    print(f"Warning: SIGTERM handler not installed: {_e}")


In [ ]:
# === Papermill parameters cell ===
# Override these from CLI: papermill ... -p RUN_ID my-run -p RESUME_FROM checkpoint_last.pth
RUN_ID = "mimic_cxr_2gpu_stage1"
RESUME_FROM = ""        # "" = auto-pick best.pth from GCS; or "checkpoint_last.pth" / "checkpoint_N.pth"
BATCH_SIZE = None       # None = giữ value trong configs/training.yaml
MAX_EPOCH = None        # None = giữ value trong configs/training.yaml
SMOKE_TEST = False      # True = override max_epoch=1, batch_size=2 cho smoke run

import os
os.environ.setdefault("RUN_ID", RUN_ID)


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle has PyTorch, torchvision, numpy, pandas, scikit-learn preinstalled.
# Install only the packages that are missing.
packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",           # latest stable; io/transform APIs are unchanged
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal",       # provides health_multimodal used by biovil_t
    "timm==0.6.13",           # 0.6.x keeps timm.models.hub API used by dist_utils.py; PyTorch 2.x compatible
    "spacy",                  # latest 3.x; stable spacy.load() API
    "nltk==3.8.1",
    "google-cloud-storage",
    "transformers==4.44.2",   # pin for Qformer.py compatibility (apply_chunking_to_forward et al.)
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# Install peft at the specific commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True
)

# Download NLTK data required by the METEOR scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Download spacy English model required by blip2.py (spacy.load("en_core_web_sm"))
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# Detect Java installation (needed for METEOR/ROUGE scoring)
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
JAVA_HOME_DETECTED = result.stdout.strip()
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

# Verify GPU count
import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("/workspace/Meta-CXR-GCP/.env"), override=False)
load_dotenv(override=False)  # fallback: CWD .env

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError(
        "WANDB_API_KEY not set. Add to .env or pass via docker -e WANDB_API_KEY=..."
    )

import wandb
wandb.login()
print("wandb: Logged in successfully")


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("/workspace/Meta-CXR-GCP/.env"), override=False)
load_dotenv(override=False)  # fallback: CWD .env

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError(
        "WANDB_API_KEY not set. Add to .env or pass via docker -e WANDB_API_KEY=..."
    )

import wandb
wandb.login()
print("wandb: Logged in successfully")


In [ ]:
import os

# Repo đã có sẵn trong Docker image; chỉ chdir.
REPO_DIR = os.environ.get("REPO_DIR", "/workspace/Meta-CXR-GCP")
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la


In [ ]:
import os

# Repo đã có sẵn trong Docker image; chỉ chdir.
REPO_DIR = os.environ.get("REPO_DIR", "/workspace/Meta-CXR-GCP")
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la


In [ ]:
import os
import yaml

with open("configs/data.yaml") as f:
    DATA_CFG = yaml.safe_load(f)

MOUNT = DATA_CFG["mount_point"]
PATHS = DATA_CFG["paths"]

assert os.path.ismount(MOUNT) or os.path.isdir(MOUNT), (
    f"GCS mount missing at {MOUNT}. Run gcsfuse before launching notebook "
    f"(see scripts/vm_startup.sh)."
)

# CSVs phải tồn tại.
for fname in DATA_CFG["required_csvs"]:
    p = os.path.join(MOUNT, fname)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing CSV on GCS mount: {p}")
print("All required CSVs present.")

# Images + reports roots.
for k in ("images_root", "reports_root"):
    p = PATHS[k]
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{k} not found: {p}")
print(f"images_root:  {PATHS['images_root']}")
print(f"reports_root: {PATHS['reports_root']}")

# Export env vars expected downstream (data/lavis loaders đọc qua env_config.yaml — set ở cell sau).
os.environ["KAGGLE_INPUT"] = MOUNT       # giữ tên biến để code cũ không phải đổi
os.environ["IMAGE_ROOT"]   = MOUNT
os.environ["REPORTS_ROOT"] = PATHS["reports_root"]
os.environ["REPORTS_CSV"]  = PATHS["reports_csv"]

import pandas as pd
df = pd.read_csv(PATHS["reports_csv"])
print(f"\nLoaded reports CSV: {len(df)} rows")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())


In [ ]:
import os
import yaml

with open("configs/data.yaml") as f:
    DATA_CFG = yaml.safe_load(f)

MOUNT = DATA_CFG["mount_point"]
PATHS = DATA_CFG["paths"]

assert os.path.ismount(MOUNT) or os.path.isdir(MOUNT), (
    f"GCS mount missing at {MOUNT}. Run gcsfuse before launching notebook "
    f"(see scripts/vm_startup.sh)."
)

# CSVs phải tồn tại.
for fname in DATA_CFG["required_csvs"]:
    p = os.path.join(MOUNT, fname)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing CSV on GCS mount: {p}")
print("All required CSVs present.")

# Images + reports roots.
for k in ("images_root", "reports_root"):
    p = PATHS[k]
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{k} not found: {p}")
print(f"images_root:  {PATHS['images_root']}")
print(f"reports_root: {PATHS['reports_root']}")

# Export env vars expected downstream (data/lavis loaders đọc qua env_config.yaml — set ở cell sau).
os.environ["KAGGLE_INPUT"] = MOUNT       # giữ tên biến để code cũ không phải đổi
os.environ["IMAGE_ROOT"]   = MOUNT
os.environ["REPORTS_ROOT"] = PATHS["reports_root"]
os.environ["REPORTS_CSV"]  = PATHS["reports_csv"]

import pandas as pd
df = pd.read_csv(PATHS["reports_csv"])
print(f"\nLoaded reports CSV: {len(df)} rows")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())


In [ ]:
import os
import subprocess
import yaml

# Java auto-detect (Docker image cài openjdk-8 ở /usr/lib/jvm/...).
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

with open("configs/data.yaml") as f:
    DATA = yaml.safe_load(f)
with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)

P = DATA["paths"]
OUTPUT_DIR = TRAIN["run"]["output_dir"]

env_config = {
    "paths": {
        "data_root":          P["data_root"],
        "mimic_cxr_jpg_root": P["mimic_cxr_jpg_root"],
        "split_csv":          P["split_csv"],
        "reports_csv":        P["reports_csv"],
        "chexpert_csv":       P["chexpert_csv"],
        "metadata_csv":       P["metadata_csv"],
        "output_dir":         OUTPUT_DIR,
        "checkpoint_dir":     OUTPUT_DIR,
    },
    "wandb": {
        "entity":  os.environ.get("WANDB_ENTITY", ""),
        "project": os.environ.get("WANDB_PROJECT", "meta-cxr"),
    },
    "java": {"home": java_home, "path": java_path},
}

os.makedirs("configs", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    yaml.safe_dump(env_config, f, sort_keys=False)

print("Written configs/env_config.yaml:")
print(yaml.safe_dump(env_config, sort_keys=False))


In [ ]:
import os
import subprocess
import yaml

# Java auto-detect (Docker image cài openjdk-8 ở /usr/lib/jvm/...).
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

with open("configs/data.yaml") as f:
    DATA = yaml.safe_load(f)
with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)

P = DATA["paths"]
OUTPUT_DIR = TRAIN["run"]["output_dir"]

env_config = {
    "paths": {
        "data_root":          P["data_root"],
        "mimic_cxr_jpg_root": P["mimic_cxr_jpg_root"],
        "split_csv":          P["split_csv"],
        "reports_csv":        P["reports_csv"],
        "chexpert_csv":       P["chexpert_csv"],
        "metadata_csv":       P["metadata_csv"],
        "output_dir":         OUTPUT_DIR,
        "checkpoint_dir":     OUTPUT_DIR,
    },
    "wandb": {
        "entity":  os.environ.get("WANDB_ENTITY", ""),
        "project": os.environ.get("WANDB_PROJECT", "meta-cxr"),
    },
    "java": {"home": java_home, "path": java_path},
}

os.makedirs("configs", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    yaml.safe_dump(env_config, f, sort_keys=False)

print("Written configs/env_config.yaml:")
print(yaml.safe_dump(env_config, sort_keys=False))


In [ ]:
import os
import subprocess
import sys
import yaml

REPO_DIR = os.environ.get("REPO_DIR", "/workspace/Meta-CXR-GCP")

with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)
with open("configs/checkpoint.yaml") as f:
    CKPT_CFG = yaml.safe_load(f)

OUTPUT_DIR = TRAIN["run"]["output_dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Resume detection: prefer best.pth từ GCS (qua gcsfuse mount) ──
from src.gcs_checkpoint import resolve_resume_checkpoint   # noqa: E402

resume_path = resolve_resume_checkpoint(
    run_id=RUN_ID,
    override=RESUME_FROM or None,
    local_output_dir=OUTPUT_DIR,
)
resume_args = ["--options", f"run.resume_ckpt_path={resume_path}"] if resume_path else []

# Smoke-test overrides (papermill param).
override_args = []
if SMOKE_TEST:
    override_args += ["--options", "run.max_epoch=1", "run.batch_size_train=2", "run.batch_size_eval=2"]
else:
    if MAX_EPOCH:
        override_args += ["--options", f"run.max_epoch={MAX_EPOCH}"]
    if BATCH_SIZE:
        override_args += ["--options", f"run.batch_size_train={BATCH_SIZE}"]

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
    "--options", f"run.output_dir={OUTPUT_DIR}",
] + resume_args + override_args

print("Launch command:")
print(" ".join(cmd))
print("\n" + "=" * 60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["RUN_ID"] = RUN_ID

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=env,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\n{'=' * 60}\nTraining exited with code: {proc.returncode}")


In [ ]:
import os
import subprocess
import sys
import yaml

REPO_DIR = os.environ.get("REPO_DIR", "/workspace/Meta-CXR-GCP")

with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)
with open("configs/checkpoint.yaml") as f:
    CKPT_CFG = yaml.safe_load(f)

OUTPUT_DIR = TRAIN["run"]["output_dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Resume detection: prefer best.pth từ GCS (qua gcsfuse mount) ──
from src.gcs_checkpoint import resolve_resume_checkpoint   # noqa: E402

resume_path = resolve_resume_checkpoint(
    run_id=RUN_ID,
    override=RESUME_FROM or None,
    local_output_dir=OUTPUT_DIR,
)
resume_args = ["--options", f"run.resume_ckpt_path={resume_path}"] if resume_path else []

# Smoke-test overrides (papermill param).
override_args = []
if SMOKE_TEST:
    override_args += ["--options", "run.max_epoch=1", "run.batch_size_train=2", "run.batch_size_eval=2"]
else:
    if MAX_EPOCH:
        override_args += ["--options", f"run.max_epoch={MAX_EPOCH}"]
    if BATCH_SIZE:
        override_args += ["--options", f"run.batch_size_train={BATCH_SIZE}"]

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
    "--options", f"run.output_dir={OUTPUT_DIR}",
] + resume_args + override_args

print("Launch command:")
print(" ".join(cmd))
print("\n" + "=" * 60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["RUN_ID"] = RUN_ID

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=env,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\n{'=' * 60}\nTraining exited with code: {proc.returncode}")


## Cell 7 — Display Evaluation Results

In [ ]:
import json
import os
import glob
import pandas as pd

OUTPUT_DIR = "/kaggle/working/output"

# ── Training logs ────────────────────────────────────────────────────────────
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
print(f"Found {len(log_files)} log file(s)")

for log_file in log_files:
    print(f"\n{'='*60}")
    print(f"Log: {log_file}")
    print('='*60)
    records = []
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(line)
    if records:
        df = pd.DataFrame(records)
        display(df)

# ── Prediction files ─────────────────────────────────────────────────────────
pred_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/predictions_*.txt", recursive=True))
print(f"\nFound {len(pred_files)} prediction file(s)")

for pred_file in pred_files[:2]:
    print(f"\n{'='*60}")
    print(f"Predictions: {pred_file}")
    print('='*60)
    with open(pred_file) as f:
        for i, line in enumerate(f):
            if i >= 10:
                print(f"  ... ({sum(1 for _ in open(pred_file))} total lines)")
                break
            print(line, end="")

# ── Checkpoint summary ───────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/**/checkpoint_*.pth", recursive=True))
print(f"\nSaved checkpoints ({len(checkpoints)}):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 ** 2)
    print(f"  {ckpt}  ({size_mb:.1f} MB)")

# ── Best checkpoint info ─────────────────────────────────────────────────────
best_ckpts = glob.glob(f"{OUTPUT_DIR}/**/checkpoint_best.pth", recursive=True)
if best_ckpts:
    print(f"\nBest checkpoint: {best_ckpts[0]}")

In [ ]:
import os
import yaml

from src.gcs_checkpoint import upload_run_to_gcs

with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)
with open("configs/checkpoint.yaml") as f:
    CKPT = yaml.safe_load(f)

upload_run_to_gcs(
    run_id=RUN_ID,
    local_dir=TRAIN["run"]["output_dir"],
    gcs_prefix=CKPT["gcs_prefix"],
    preserve_files=CKPT["preserve_files"],
    keep_last_n=CKPT["keep_last_n_epoch_ckpts"],
    verify=CKPT["upload"]["verify_after_upload"],
)
print("Checkpoint sync to GCS complete.")


In [ ]:
import os
import yaml

from src.gcs_checkpoint import upload_run_to_gcs

with open("configs/training.yaml") as f:
    TRAIN = yaml.safe_load(f)
with open("configs/checkpoint.yaml") as f:
    CKPT = yaml.safe_load(f)

upload_run_to_gcs(
    run_id=RUN_ID,
    local_dir=TRAIN["run"]["output_dir"],
    gcs_prefix=CKPT["gcs_prefix"],
    preserve_files=CKPT["preserve_files"],
    keep_last_n=CKPT["keep_last_n_epoch_ckpts"],
    verify=CKPT["upload"]["verify_after_upload"],
)
print("Checkpoint sync to GCS complete.")
